In [12]:
from typing import TypedDict

class ReviewState(TypedDict):
    generated_text: str

In [13]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

def review_node(state: ReviewState):
    # 사용자(검토자)에게 생성된 콘텐츠를 편집하도록 요청
    updated = interrupt({
        "instruction": "이 콘텐츠를 검토하고 편집하세요",
        "content": state["generated_text"],
    })
    return {"generated_text": updated}


graph_builder = StateGraph(ReviewState)

graph_builder.add_node("review", review_node)
graph_builder.add_edge(START, "review")
graph_builder.add_edge("review", END)

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)


In [14]:
config = {"configurable": {"thread_id": "review-1"}}
initial = graph.invoke({"generated_text": "초기 초안"}, config=config)

In [15]:
initial["__interrupt__"]

[Interrupt(value={'instruction': '이 콘텐츠를 검토하고 편집하세요', 'content': '초기 초안'}, id='79c06e0bdb037f559a8eb27a63f437a5')]

In [16]:
state = graph.get_state(config)

In [17]:
state

StateSnapshot(values={'generated_text': '초기 초안'}, next=('review',), config={'configurable': {'thread_id': 'review-1', 'checkpoint_ns': '', 'checkpoint_id': '1f10d2ee-a2e2-686f-8000-9b7cdad7554c'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-02-19T01:04:17.374628+00:00', parent_config={'configurable': {'thread_id': 'review-1', 'checkpoint_ns': '', 'checkpoint_id': '1f10d2ee-a2e1-645c-bfff-dc3f19a8a7af'}}, tasks=(PregelTask(id='91684379-1983-c35a-725a-f0d5884a28f9', name='review', path=('__pregel_pull', 'review'), error=None, interrupts=(Interrupt(value={'instruction': '이 콘텐츠를 검토하고 편집하세요', 'content': '초기 초안'}, id='79c06e0bdb037f559a8eb27a63f437a5'),), state=None, result=None),), interrupts=(Interrupt(value={'instruction': '이 콘텐츠를 검토하고 편집하세요', 'content': '초기 초안'}, id='79c06e0bdb037f559a8eb27a63f437a5'),))

In [19]:
prompt = state.interrupts[0].value
print(prompt)
user_input = input(" > ") # 재개(resume)를 위해 사용자 입력 받기

{'instruction': '이 콘텐츠를 검토하고 편집하세요', 'content': '초기 초안'}


In [20]:
# 사용자(검토자)가 편집한 텍스트로 재개
final_state = graph.invoke(
    # Command(resume="검토 후 개선된 초안"),
    Command(resume=user_input),
    config=config,
)

In [21]:
final_state

{'generated_text': '초기 초안 수정'}